# Prespecified positional sensitivity in 5X72

Question: how do the fixed score and declared protein-pair distances change when
all 39 atoms move by the seven prespecified Cartesian offsets, with internal
geometry and each arm's H fixed?

This human-executable analysis reads the original 112 evaluations. It performs
no search, optimization, preparation, H regeneration or new scoring. Recorded
values are observations, not an affinity ranking or a validated scientific model.
The [prescription](https://github.com/uibcdf/dockingmt/issues/17#issuecomment-6094392354)
precedes the original producer. Software qualification is recorded separately.


In [1]:
import gzip, hashlib, itertools, json
from pathlib import Path
import numpy as np
import pandas as pd
import dockingmt as dmt

checkout = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'devtools/qualify_5x72_displacement.py').is_file())
directory = checkout / 'devguide/validation/data/5x72_displacement'
payload = (directory / 'audit_2026-10-09.json.gz').read_bytes()
sha = lambda data: hashlib.sha256(data).hexdigest()
assert sha(payload) == 'afb8bfa4092abc78f8cd2daf33da7248fe39fbd1856b23e24a192f4813fb72a8'
assert sha(gzip.decompress(payload)) == '0bbf5990234cec5b739d7f32f00e035ada42d9bd3277932eab831c9f3545226c'
record = json.loads(gzip.decompress(payload))
assert sha((directory / 'producer_2026-10-09.py.txt').read_bytes()) == record['consumer_source_sha256']['devtools/qualify_5x72_displacement.py']
assert record['unit_contract'] == dict(coordinates='angstrom', translation='angstrom', pair_distance='angstrom', rmsd='angstrom', scores='kcal/mol')
assert len(record['points']) == 28 and record['new_fixed_evaluations'] == 112
assert record['fresh_zero_anchor_evaluations'] == 16
assert record['new_nonzero_translation_evaluations'] == 96
assert record['new_searches'] == record['new_optimizations'] == 0
assert record['offsets_angstrom'] == [[0,0,0],[-.25,0,0],[.25,0,0],[0,-.25,0],[0,.25,0],[0,0,-.25],[0,0,.25]]
pairs = np.asarray(list(itertools.combinations(range(39), 2)))
evaluations = 0
for point in record['points']:
    geometry = point['geometry']
    coordinates = np.asarray(geometry['full39_coordinates_angstrom'])
    zero = next(p for p in record['points'] if p['ligand'] == point['ligand'] and p['geometry_arm'] == point['geometry_arm'] and p['offset_index'] == 0)
    origin = np.asarray(zero['geometry']['full39_coordinates_angstrom'])
    np.testing.assert_allclose(coordinates-origin, np.broadcast_to(record['offsets_angstrom'][point['offset_index']], (39,3)), rtol=0, atol=1e-10)
    np.testing.assert_array_equal(geometry['all741_pair_indices'], pairs)
    np.testing.assert_allclose(geometry['all741_translated_distances_angstrom'], np.linalg.norm(coordinates[pairs[:,0]]-coordinates[pairs[:,1]],axis=1), rtol=0, atol=1e-10)
    np.testing.assert_allclose(geometry['all741_original_distances_angstrom'], geometry['all741_translated_distances_angstrom'], rtol=0, atol=1e-10)
    for order, contexts in point['fixed_scores'].items():
        assert point['poses'][order]['rank'] is None and point['poses'][order]['scores'] == {}
        for context, score in contexts.items():
            history = score['metadata']['scoring_history']
            assert len(history) == 1 and history[0]['operation'] == 'score'
            assert len(history[0]['scores']) == 8 and 'vina' not in score['scores']
            assert dmt.verify_captured_inputs(history[0]['backend_artifacts'])
            evaluations += 1
assert evaluations == 112
print('Authenticated 28 full-atom points / 112 captures / 20,748 pair distances; no new scoring.')


Authenticated 28 full-atom points / 112 captures / 20,748 pair distances; no new scoring.


## Full seven-point score curves

Each row compares one ligand, coordinate arm and receptor context to its own
zero anchor. Both ROOT orders must agree exactly. Offset indices are fixed by
the prescription, not chosen after observing a favorable score. Units: kcal/mol.


In [2]:
rows = []
for summary in record['summaries']:
    values = np.asarray(summary['seven_totals_kcal_mol'])
    assert summary['finite_minimum_offset_index'] == int(np.argmin(values))
    assert summary['finite_maximum_offset_index'] == int(np.argmax(values))
    assert summary['nonzero_offsets_with_strictly_lower_reported_total'] == int(np.sum(values[1:] < values[0]))
    counterpart = next(s for s in record['summaries'] if s['ligand']==summary['ligand'] and s['geometry_arm']==summary['geometry_arm'] and s['context']==summary['context'] and s['order']!=summary['order'])
    assert summary['seven_totals_kcal_mol'] == counterpart['seven_totals_kcal_mol']
    for i, delta in enumerate(summary['all_component_deltas_kcal_mol']):
        assert delta['fixed'] == values[i]-values[0]
    if summary['order']=='native':
        rows.append({'ligand':summary['ligand'], 'arm':summary['geometry_arm'], 'context':summary['context'], **{f'offset {i}':float(v) for i,v in enumerate(values)}, 'lower nonzero':summary['nonzero_offsets_with_strictly_lower_reported_total'], 'minimum delta':summary['finite_minimum_minus_zero_kcal_mol']})
display(pd.DataFrame(rows))


,ligand,arm,context,offset 0,offset 1,offset 2,offset 3,offset 4,offset 5,offset 6,lower nonzero,minimum delta
0,p59,original_rigid,sham,-7.618,-8.304,-6.582,-7.625,-7.055,-6.274,-8.408,3,-0.790
1,p59,original_rigid,occupied,-8.551,-9.170,-7.347,-8.460,-8.062,-7.781,-8.098,1,-0.619
2,p59,experimental_reference,sham,-9.661,-9.380,-9.370,-8.954,-9.521,-8.223,-10.388,1,-0.727
3,p59,experimental_reference,occupied,-10.534,-10.176,-10.340,-9.821,-10.412,-9.417,-10.478,0,0.000
4,p69,original_rigid,sham,-1.666,-4.178,2.034,-0.718,-2.043,-2.863,0.064,3,-2.512
5,p69,original_rigid,occupied,-2.667,-5.221,1.091,-1.798,-2.975,-3.185,-1.280,3,-2.554
6,p69,experimental_reference,sham,-9.626,-10.252,-8.280,-8.490,-9.988,-9.520,-9.152,2,-0.626
7,p69,experimental_reference,occupied,-10.499,-11.223,-9.076,-9.380,-10.855,-9.612,-10.346,2,-0.724


## Prespecified pair distances and interpretation

Distances use the exact submitted three-decimal PDBQT projections, whereas RMSD
and internal geometry checks use the unrounded 39-atom system. Keep these axes
separate. The table retains every selected pair and offset; neither a short pair
nor a score change assigns an atom-pair energy or a causal explanation.


In [3]:
pair_rows = []
for point in record['points']:
    assert point['selected_pair_distances']['sham'] == point['selected_pair_distances']['occupied']
    table = point['selected_pair_distances']['sham']
    for pair in table['declared_pairs']:
        atom = pair['receptor_atom']
        pair_rows.append({'ligand':point['ligand'], 'arm':point['geometry_arm'], 'offset':point['offset_index'], 'source index':pair['source_atom_index'], 'protein atom':f"{atom['chain']} {atom['group']}{atom['group_id']} {atom['name']}/{atom['type']}", 'distance A':pair['distance_angstrom'], 'unrounded heavy RMSD A':point['geometry']['heavy_positional_rmsd_angstrom']})
assert len(pair_rows) == 56
display(pd.DataFrame(pair_rows))
print(record['interpretation'])


,ligand,arm,offset,source index,protein atom,distance A,unrounded heavy RMSD A
0,p59,original_rigid,0,16,A ARG61 NH1/N,2.389969,5.131376e-01
1,p59,original_rigid,0,16,A GLN78 NE2/N,2.439326,5.131376e-01
2,p59,experimental_reference,0,16,A ARG61 NH1/N,2.537896,6.393831e-15
3,p59,experimental_reference,0,16,A GLN78 NE2/N,2.950850,6.393831e-15
4,p69,original_rigid,0,4,A GLN116 OE1/OA,2.223982,6.794519e-01
5,p69,original_rigid,0,16,A TYR149 OH/OA,2.420689,6.794519e-01
6,p69,experimental_reference,0,4,A GLN116 OE1/OA,2.942091,1.281975e-15
7,p69,experimental_reference,0,16,A TYR149 OH/OA,2.796309,1.281975e-15
8,p59,original_rigid,1,16,A ARG61 NH1/N,2.529219,5.707978e-01
9,p59,original_rigid,1,16,A GLN78 NE2/N,2.529587,5.707978e-01


Finite positional sensitivity, not optimization, returned-pose recovery, pair-energy decomposition or causal attribution. H is fixed within each arm and remains different between arms.


## What this permits us to conclude

The seven-point population measures finite positional sensitivity. Some translated
P69 original-rigid points score lower; the P59 earlier-reference occupied arm has
no lower nonzero point in this population. Both observations are retained.

These are constructed fixed evaluations, not recovered docking poses, local
optima, convergence or affinity estimates. H stays fixed within each arm but
differs between arms. No torsion, preparation, ROOT, ranking or contact-based
remedy is selected. The original raw archive and producer bytes are unchanged.
